# AtomicNG — Quickstart

**Atomic N-gram Memory**: forward-only recording knowledge in n-gram memory LLM
(method [arXiv:2610.31125](https://arxiv.org/abs/2610.31125)).
`T = T0 + Δ` — the base table is frozen, new knowledge is accumulated in a sparse patch Δ;
Clearing Δ restores the original model **bit for bit**.

The core is pure NumPy (CPU, no ML frameworks); torch is only needed for writers and HF.

In [ ]:
# Installation (in repo): pip install -e ".[dev]"  ·  PyPI: Pip install atomicng
import atomicng
print(atomicng.__version__)

## 1. Write, read, and verify exact rollback

In [ ]:
model = atomicng.TinyModel()  # CI backend; for real models: atomicng.backends.hf.attach_sidecar
undo = atomicng.UndoJournal()             # Exact block rollback using a pre-image journal
audit = atomicng.AuditLog("audit.jsonl")  # append-only journal of operations

report = atomicng.write(
    model,
    "AtomicNG stores new knowledge in a sparse patch Δ on top of a frozen table.",
    audit=audit,
    undo=undo,
    source="demo-1",
)
print(report.summary())
print(model.memory.stats())

# Check that clearing Δ restores base outputs bit for bit.
verdict = atomicng.verify_deletion(model)
print(verdict.message)

## 2. Write budgets and caps

In [ ]:
policies = atomicng.WritePolicies(row_budget=5, hot_row_norm_cap=1.0)
model2 = atomicng.TinyModel()
try:
    for i in range(10):
        atomicng.write(model2, "document %d about row budgets and guards" % i, policies=policies)
except atomicng.BudgetExceeded as exc:
    print("Budget rejected the entire write:", exc)

## 3. Evaluation protocols: acquisition, distraction, and chain

In [ ]:
docs = ["Document %d about topic-%d with specific facts and numbers %d." % (i, i, i) for i in range(12)]
model3 = atomicng.TinyModel()
suite = atomicng.run_suite(model3, docs, chain_steps=8)
print("acquisition:", suite["acquisition"])
print("chain: drift", suite["chain"]["final_drift"], "recall", suite["chain"]["final_recall"])

## 4. Monitoring and publication

In [ ]:
# Check verification, neutral drift, and hot rows before publication.
gate = atomicng.publish_check(model3)
print("publish:", gate.passed, gate.checks)

# Render or download the standalone HTML report.
html_report = atomicng.build_report(model3.memory)
print("report:", len(html_report), "byte")

## Next
- Paper conformance — `FIDELITY.md`, conformance tests: `pytest tests/test_fidelity.py`
- Real models: `atomicng.backends.hf.attach_sidecar` + `scripts/train_writer.py`
- CLI: `atomicng write / eval / bench / monitor / export-ple / publish / smoke`